# Delta Lake Basics

Goal: store data in Delta tables and use their core features: schema checks, data changes, history and time travel.

**Learning objectives**
- Explain what Delta Lake is: Parquet data files plus a `_delta_log` transaction log
- Create a Delta table and explain schema enforcement and schema evolution
- Change data with `INSERT`, `UPDATE`, `DELETE` and `MERGE INTO`
- Read table history, query old versions and undo a mistake with `RESTORE`
- Distinguish managed and external tables and recover a dropped table with `UNDROP`

**Prerequisites:** the setup notebook, run with `%run ../setup/00_setup` in section 1.

## 1. Setup

The setup notebook defines `CATALOG`, the schema names and `DATASET_PATH` used below.

> **Note:** everything after the Summary is ⊕ BONUS material for self-study.

In [ ]:
%run ../setup/00_setup

### Configuration

Import the libraries, show your catalog and schemas, and set them as the default.

In [ ]:
# Imports used in this notebook
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta

In [ ]:
# Show your catalog and schemas
display(
    spark.createDataFrame([
        (CATALOG, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA)
    ], ['catalog', 'bronze_schema', 'silver_schema', 'gold_schema'])
)

In [ ]:
# Set catalog and schema as default
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

## 2. Delta Lake Core Features

Delta Lake is the default table format in Databricks. It is a transactional layer over Parquet files that adds ACID transactions (Atomicity, Consistency, Isolation, Durability), schema enforcement and time travel.

> **Note:** every table you create in Databricks is a Delta table by default. Delta = Parquet files + transaction log (`_delta_log`).

![Delta Lake Core Features](../../assets/images/8850570fe2b147eb86cb690d51bc798c.png)

Every operation on a Delta table is recorded in the Delta Log: the `_delta_log/` folder with JSON files that describe each transaction.

- ACID transactions: every operation completes fully or not at all
- Schema enforcement: writes that do not match the table schema are rejected
- Schema evolution: new columns are added in a controlled way, without breaking existing pipelines
- Constraints: data quality rules at table level (identity columns and CHECK constraints are in the ⊕ BONUS section)

Typical uses: transactional updates in a data lake, data quality checks, one table for batch and streaming workloads.

### Create the First Delta Table

Load a CSV file from a Unity Catalog Volume (governed storage for files), save it as a managed Delta table, then look at the data.

In [ ]:
# Load customer data from Unity Catalog Volume
customers_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATASET_PATH}/customers/customers.csv")
)

In [ ]:
# Drop the table if it exists from an earlier run
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.customers_delta")

In [ ]:
# Create managed Delta table
# SQL equivalent: CREATE OR REPLACE TABLE customers_delta AS
#                 SELECT * FROM read_files('<DATASET_PATH>/customers/customers.csv', format => 'csv', header => true)
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")

In [ ]:
# Show the first rows of the new table
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta").limit(5))

### Schema Enforcement in Action

Schema enforcement prevents "garbage in, garbage out". Delta compares the schema of incoming data with the table schema and rejects incompatible writes.

![Example: Schema Enforcement in Action](../../assets/images/3c274f5e758047b2b8033f29fd179c85.png)

| Incoming DataFrame | What Delta does on `append` |
|---|---|
| Extra column not in the table | Rejected — `DELTA_METADATA_MISMATCH` |
| Column type that cannot be converted safely | Rejected |
| Missing column | Accepted — the column is filled with `NULL` |
| Different type, e.g. a STRING value for a `DATE` column | Cast explicitly before the append (code below) |

The demo below sends a row with an extra column, the case Delta always blocks.

In [ ]:
# Check current table schema
spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta").printSchema()

In [ ]:
# Incoming row with an EXTRA column (loyalty_level) that the table does not have
invalid_data = spark.createDataFrame([
    ("CUST999999", "Test", "Customer", "invalid_email", "+48 123 456 789", "2025-11-28", "Gold")
], ["customer_id", "first_name", "last_name", "email", "phone", "registration_date", "loyalty_level"])

invalid_data.printSchema()

In [ ]:
# Try to append the row with the extra column
rows_before = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta").count()

try:
    invalid_data.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")
    print("Write succeeded — schema enforcement did NOT block it")
except Exception as e:
    print("Schema enforcement rejected the write:")
    print(str(e)[:250])

rows_after = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta").count()
print(f"\nRows before: {rows_before:,} | Rows after: {rows_after:,} — the table is unchanged")

### Fix the Data and Append Again

Drop the extra column and cast the date. The variable keeps its name `invalid_data`, but after this cell it holds valid data.

In [ ]:
# Fix the incoming data: drop the column the table does not know, cast the date explicitly.
# Missing columns (city, state, country, customer_segment) are allowed — they will be NULL.
# SQL equivalent of the cast: CAST(registration_date AS DATE)
from pyspark.sql.functions import col
from pyspark.sql.types import DateType

invalid_data = (
    invalid_data
    .drop("loyalty_level")
    .withColumn("registration_date", col("registration_date").cast(DateType()))
)

In [ ]:
# Check the fixed schema and row
invalid_data.printSchema()
display(invalid_data)

In [ ]:
# Append the fixed row — this write succeeds
invalid_data.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")

### Schema Evolution

Schema evolution adds new columns to an existing Delta table without breaking the code that uses it.
With the `mergeSchema` option, Delta adds the new columns from the incoming data automatically.

In [ ]:
# Data with additional column (customer_tier)
extended_customers = spark.createDataFrame([
    ("CUST010001", "New", "Customer", "new@example.com", "+48 111 222 333", "Warsaw", "MZ", "Poland", "2023-12-01", "Basic", "Premium"),
    ("CUST010002", "Another", "Customer", "another@example.com", "+48 444 555 666", "Krakow", "MP", "Poland", "2023-12-02", "Premium", "Standard")
], ["customer_id", "first_name", "last_name", "email", "phone", "city", "state", "country", "registration_date", "customer_segment", "customer_tier"])
display(extended_customers)

In [ ]:
# Current table, before schema evolution
df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")
display(df.orderBy("customer_id"))

In [ ]:
# Cast registration_date to proper type
from pyspark.sql.functions import col
from pyspark.sql.types import DateType

extended_customers = extended_customers.withColumn("registration_date", col("registration_date").cast(DateType()))

In [ ]:
# Enable automatic schema evolution with mergeSchema option
# SQL equivalent: ALTER TABLE customers_delta ADD COLUMN customer_tier STRING, then INSERT INTO ... (ALTER TABLE: see "Add a Column with ALTER TABLE"; INSERT: see section 3)
extended_customers.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")

In [ ]:
# Check new schema - notice the new customer_tier column
spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta").printSchema()

In [ ]:
# Verify data - the new rows have customer_tier filled (older rows have NULL)
display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")
    .select("customer_id", "first_name", "last_name", "customer_tier")
    .filter(col("customer_id") == "CUST010001")
)

### Add a Column with ALTER TABLE

`mergeSchema` adds columns as a side effect of a write. To add a column on purpose, without writing data, use DDL (data definition language: statements such as `ALTER TABLE`).
Existing rows get `NULL` in the new column; the change is a new version in `DESCRIBE HISTORY`.

In [ ]:
# Add a column explicitly — existing rows get NULL
spark.sql(f"ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.customers_delta ADD COLUMN loyalty_points INT")

In [ ]:
# Show the new column
display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")
    .select("customer_id", "customer_tier", "loyalty_points")
    .limit(5)
)

## 3. CRUD Operations & MERGE

Delta tables support CRUD: Create, Read, Update, Delete. Every operation is:
- atomic: it completes fully or is rolled back
- ACID-compliant: data stays consistent
- recorded in the Delta Log: a full audit trail of changes

`MERGE INTO` (also called upsert) combines INSERT and UPDATE in one atomic transaction. It is essential for CDC (Change Data Capture: loading the changes made in a source system).

### INSERT

Add new records to an existing table.

In [ ]:
# INSERT new customers
spark.sql(f"""
    INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.customers_delta
    (customer_id, first_name, last_name, email, phone, city, state, country, registration_date, customer_segment, customer_tier)
    VALUES 
        ('CUST020001', 'Insert', 'Customer1', 'insert1@example.com', '+48 111 111 111', 'Warsaw', 'MZ', 'Poland', '2023-12-10', 'Premium', 'Gold'),
        ('CUST020002', 'Insert', 'Customer2', 'insert2@example.com', '+48 222 222 222', 'Gdansk', 'PM', 'Poland', '2023-12-11', 'Basic', 'Silver')
""")

In [ ]:
# Verify insertion
display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")
    .filter(F.col("customer_id").like("CUST02%"))
    .orderBy("customer_id")
)

### UPDATE

Change existing records in the table.

In [ ]:
# UPDATE customer tier for specific customers
spark.sql(f"""
    UPDATE {CATALOG}.{BRONZE_SCHEMA}.customers_delta
    SET customer_tier = 'Platinum'
    WHERE customer_id IN ('CUST010001', 'CUST020001')
""")

In [ ]:
# Verify update
display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")
    .filter(F.col("customer_tier") == "Platinum")
)

### DELETE

Remove records from a Delta table.

In [ ]:
# DELETE specific customer
spark.sql(f"""
    DELETE FROM {CATALOG}.{BRONZE_SCHEMA}.customers_delta
    WHERE customer_id = 'CUST020002'
""")

In [ ]:
# Verify deletion
deleted_check = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta") \
    .filter(F.col("customer_id") == "CUST020002") \
    .count()

display(
    spark.createDataFrame([
        ("Records with customer_id CUST020002", deleted_check)
    ], ["description", "count"])
)

### MERGE INTO (Upsert)

MERGE updates existing records and inserts new ones in one atomic transaction. It is common when you process changes from transactional systems (CDC):
- update records when a match is found
- insert records when no match exists
- delete records based on conditions (optional)

![Example: MERGE INTO (Upsert)](../../assets/images/ff01677d3d4a45d6a6a7530d8911b785.png)

MERGE INTO syntax:

```sql
MERGE INTO target
USING source ON target.key = source.key
WHEN MATCHED THEN UPDATE SET ...
WHEN NOT MATCHED THEN INSERT (...)  VALUES (...)
WHEN NOT MATCHED BY SOURCE THEN DELETE  -- optional
```

| Clause | Description |
|---|---|
| `WHEN MATCHED` | Executed when a source row matches a target row (UPDATE or DELETE) |
| `WHEN NOT MATCHED` | Executed when the source row has no match in target (INSERT) |
| `WHEN NOT MATCHED BY SOURCE` | Executed when the target row has no match in source (DELETE) |
| `AND condition` | Optional extra filter on any `WHEN` clause |

In [ ]:
# Prepare data for merge (mix of updates and new records)
merge_data = spark.createDataFrame([
    ("CUST010001", "Updated", "Name", "updated@example.com", "+48 999 999 999", "Poznan", "WP", "Poland", "2023-12-01", "VIP", "Diamond"),  # Update existing
    ("CUST030001", "Brand", "New", "brand.new@example.com", "+48 777 777 777", "Wroclaw", "DS", "Poland", "2023-12-15", "Basic", "Bronze"),   # Insert new
    ("CUST030002", "Another", "New", "another.new@example.com", "+48 888 888 888", "Lodz", "LD", "Poland", "2023-12-16", "Premium", "Silver") # Insert new
], ["customer_id", "first_name", "last_name", "email", "phone", "city", "state", "country", "registration_date", "customer_segment", "customer_tier"])

In [ ]:
# Register the data as a temporary view for the MERGE
merge_data.createOrReplaceTempView("customer_updates")

In [ ]:
# MERGE INTO operation (Upsert): update matched customers, insert new ones
spark.sql(f"""
    MERGE INTO {CATALOG}.{BRONZE_SCHEMA}.customers_delta AS target
    USING customer_updates AS source
    ON target.customer_id = source.customer_id
    WHEN MATCHED THEN
        UPDATE SET
            first_name = source.first_name, last_name = source.last_name,
            email = source.email, phone = source.phone,
            city = source.city, state = source.state, country = source.country,
            customer_segment = source.customer_segment, customer_tier = source.customer_tier
    WHEN NOT MATCHED THEN
        INSERT (customer_id, first_name, last_name, email, phone, city, state, country, registration_date, customer_segment, customer_tier)
        VALUES (source.customer_id, source.first_name, source.last_name, source.email, source.phone, source.city, source.state, source.country, source.registration_date, source.customer_segment, source.customer_tier)
""")

In [ ]:
# Verify MERGE results
display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")
    .filter(F.col("customer_id").isin(["CUST010001", "CUST030001", "CUST030002"]))
    .orderBy("customer_id")
)

## 4. Metadata and Analytics

Delta keeps metadata about each table and every operation on it. You use it for:
- auditing: who changed what, and when
- debugging: operation metrics, such as how many rows changed
- compliance: meeting regulatory requirements for data lineage

| Command | Signature | Description |
|---|---|---|
| `DESCRIBE DETAIL` | `DESCRIBE DETAIL table` | Returns file count, size, location, partitioning, table properties |
| `DESCRIBE HISTORY` | `DESCRIBE HISTORY table [LIMIT n]` | Full audit log: operation, user, timestamp, metrics per version |
| `SHOW TBLPROPERTIES` | `SHOW TBLPROPERTIES table` | Lists all key-value table properties (e.g., CDF enabled, constraints) |
| `DESCRIBE EXTENDED` | `DESCRIBE EXTENDED table` | Schema + storage info + table properties in one output |

### DESCRIBE DETAIL

Show the table metadata and its physical storage: location, number of files, size.

In [ ]:
# Detailed table information
display(
    spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.customers_delta")
)

### DESCRIBE HISTORY

Every change creates a new table version. `DESCRIBE HISTORY` shows the operation and its metrics for each version.

In [ ]:
# History with additional metrics — operationMetrics holds different keys per operation
# (WRITE: numOutputRows, DELETE: numDeletedRows, MERGE: numTargetRowsInserted/Updated, ...)
history_df = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers_delta")

display(
    history_df.select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    )
)

## 5. Time Travel and RESTORE

Time travel reads earlier versions of a table. Every change creates a new version in the transaction log (`_delta_log`).
The old data files stay in storage, so older versions remain readable until `VACUUM` deletes those files after the retention period.

You can read a version number (`VERSION AS OF`) or a point in time (`TIMESTAMP AS OF`), compare data between versions for audit, and roll a table back with `RESTORE`.

![Time Travel and RESTORE](../../assets/images/7e9c029eeeb14e44b5ce68c5af90f350.png)

| Command | Signature | Description |
|---|---|---|
| `VERSION AS OF` | `SELECT * FROM table VERSION AS OF n` | Reads data at Delta version number `n` |
| `TIMESTAMP AS OF` | `SELECT * FROM table TIMESTAMP AS OF '2024-01-01'` | Reads data as it was at the given timestamp |
| `RESTORE` | `RESTORE TABLE table TO VERSION AS OF n` | Rolls the table back to version `n` (creates a new version in history). Works only for versions inside the retention window |
| `VACUUM` | `VACUUM table` | Deletes data files that the current version no longer uses, once they are older than the retention period (default 7 days). `_delta_log` is not touched |

### Prepare a Demo Table

Create a small table and change it step by step. Each change is a new version.

In [ ]:
# Drop the demo table if it exists from an earlier run
spark.sql(f"""DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo """)

In [ ]:
# Create a new table specifically for Time Travel demonstration
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo (
    id INT,
    name STRING,
    status STRING,
    updated_at TIMESTAMP
) USING DELTA
""")

### A Helper for Version Numbers

Databricks can add automatic `OPTIMIZE` versions (`OPTIMIZE` rewrites many small files into fewer, larger ones).
So we read version numbers from the history instead of counting them. The helper below does that.

In [ ]:
# Helper: find the newest version that changed the data
def latest_version(table):
    """Newest version that changed the data. Databricks may add automatic OPTIMIZE
    commits after a write, so version numbers are read from the history, not assumed."""
    return (spark.sql(f"DESCRIBE HISTORY {table}")
            .filter("operation != 'OPTIMIZE'")
            .agg(F.max("version"))
            .first()[0])

In [ ]:
# Insert initial data (CREATE TABLE itself is version 0 — an empty table)
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VALUES
    (1, 'Alice', 'active', current_timestamp()),
    (2, 'Bob', 'active', current_timestamp()),
    (3, 'Charlie', 'active', current_timestamp())
""")

v_initial = latest_version(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")
print(f"Version {v_initial}: initial data inserted")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo"))

In [ ]:
# Update some records
spark.sql(f"""
UPDATE {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo
SET status = 'premium', updated_at = current_timestamp()
WHERE name = 'Alice'
""")

v_alice = latest_version(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")
print(f"Version {v_alice}: Alice upgraded to premium")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo"))

In [ ]:
# Insert new record
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VALUES
    (4, 'Diana', 'new', current_timestamp())
""")

v_diana = latest_version(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")
print(f"Version {v_diana}: Diana added")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo"))

In [ ]:
# Delete a record
spark.sql(f"""
DELETE FROM {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo
WHERE name = 'Charlie'
""")

v_charlie = latest_version(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")
print(f"Version {v_charlie}: Charlie deleted")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo"))

### Table History

Use `DESCRIBE HISTORY` to see all operations on the table.

> **Note:** `OPTIMIZE` rows you did not run are the automatic compactions explained before the helper above (`operationParameters.auto = true`).

In [ ]:
# Show complete history of all operations
display(
    spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")
)

> **Note:** on Unity Catalog managed tables, predictive optimization (Databricks runs table maintenance for you) runs
> `OPTIMIZE` (compacts small files), `VACUUM` (deletes old data files) and `ANALYZE` (collects statistics for the query planner).
> It is on by default for accounts created since 11 November 2024 and is being enabled gradually for older accounts.

> **Tip:** for the data layout of new tables, Databricks recommends liquid clustering (`CLUSTER BY AUTO` or `CLUSTER BY (col)`),
> which keeps rows with similar values in the same files. It replaces partitioning (one folder per column value)
> and `ZORDER` (an older way to sort data inside files).

### Time Travel Queries

Read previous versions of the data with `VERSION AS OF`. `TIMESTAMP AS OF` works the same way with a timestamp.

In [ ]:
# Access the initial state (after the first INSERT)
print(f"Version {v_initial} - Initial data (before any changes):")
display(
    spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VERSION AS OF {v_initial}")
)

In [ ]:
# Access the state after the Alice upgrade
print(f"Version {v_alice} - After Alice upgrade:")
display(
    spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VERSION AS OF {v_alice}")
)

In [ ]:
# Compare record counts between versions
steps = [
    ("CREATE TABLE (empty)", 0),
    ("Initial insert",        v_initial),
    ("Alice upgraded",        v_alice),
    ("Diana added",           v_diana),
    ("Charlie deleted",       v_charlie),
]

version_counts = []
for step, v in steps:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VERSION AS OF {v}").first()[0]
    version_counts.append((step, v, count))

display(spark.createDataFrame(version_counts, ["step", "version", "record_count"]))

### Undo an Accidental Delete with RESTORE

Simulate an accidental delete, then roll the table back with `RESTORE`.

> **Warning:** RESTORE works only while the old version is inside the retention window. It undoes recent mistakes; it is not a backup.

In [ ]:
# Accident: DELETE without WHERE removes ALL rows
spark.sql(f"DELETE FROM {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")

print("Oh no! All data deleted!")
print("Record count after deletion:", spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo").count())

In [ ]:
# Check history to find the last good version
history = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")
display(history.select("version", "timestamp", "operation"))

In [ ]:
# RESTORE to the version before the accidental deletion.
# Every commit records the version it started from (readVersion) —
# for the last DELETE that is exactly the last good state.
last_good_version = (
    spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")
    .filter("operation = 'DELETE'")
    .orderBy(F.desc("version"))
    .first()["readVersion"]
)

In [ ]:
# Roll the table back to the last good version
print(f"Restoring to version: {last_good_version}")
spark.sql(f"RESTORE TABLE {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo TO VERSION AS OF {last_good_version}")

In [ ]:
# Verify restoration
print("Data restored successfully!")
print("Record count after RESTORE:", spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo").count())
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo"))

In [ ]:
# History now shows RESTORE as a new version
display(
    spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")
)

### VACUUM and Retention

- Old data files stay in storage so that time travel works. `VACUUM` deletes the files that the current version no longer uses, once they are older than the retention period: table property `delta.deletedFileRetentionDuration`, default 7 days.
- `VACUUM` removes data files only, not the history in `_delta_log`. Versions whose files are gone can no longer be read or restored.
- On Unity Catalog managed tables, predictive optimization (where enabled) runs `VACUUM` for you.
```sql
VACUUM customers_delta   -- uses the table's retention setting
```

> **Warning:** time travel is not a backup. It undoes recent mistakes, within the retention window.

A hands-on demo with 0-hour retention is in the ⊕ BONUS section at the end of this notebook.

## 6. Managed vs External Tables

In Unity Catalog there are two types of Delta tables. The difference shows when you drop a table.

| Feature | Managed table | External table |
|---------|------------------|-------------------|
| Data location | Managed by Databricks (default location in Unity Catalog) | Specified by the user (`LOCATION 'abfss://...'`) |
| `DROP TABLE` | Removes the table from the catalog; the data files are purged after the recovery period (default 7 days). Until then `UNDROP TABLE` brings it back | Removes only the catalog entry — files remain in your storage |
| Lifecycle management | Databricks (predictive optimization runs maintenance) | You |
| When to use | Most cases — simpler management | Data shared with other systems, external data, migration |

### Managed Table (default)

If you do not specify `LOCATION`, you create a managed table.

In [ ]:
# Create a managed table — no LOCATION, Databricks manages the data location
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.managed_demo
    (id INT, name STRING, created_at TIMESTAMP)
    USING DELTA
    COMMENT 'This is a Managed Table — Databricks controls the file location'
""")

In [ ]:
# Check where the files are stored
spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.managed_demo").display()

### Drop and Recover a Managed Table (UNDROP)

`DROP TABLE` on a Unity Catalog managed table removes it from the catalog, but the data files are kept for a recovery period (default 7 days).
During that period `UNDROP TABLE` brings the table back with its data and history; afterwards the files are purged automatically.

`SHOW TABLES DROPPED` (Public Preview) lists the tables you can still recover in the current schema.

In [ ]:
%sql
-- Drop the managed table (the Configuration cell set the current catalog and schema)
DROP TABLE managed_demo

In [ ]:
%sql
-- Recover it: possible for 7 days after the DROP (default recovery period)
UNDROP TABLE managed_demo

In [ ]:
%sql
-- Check that the table is back
DESCRIBE DETAIL managed_demo

### Check the Table Type

`information_schema.tables` shows the type of every table in your schema.

In [ ]:
# Check table type programmatically
spark.sql(f"""
    SELECT table_schema,table_name,table_type
    FROM {CATALOG}.information_schema.tables
    WHERE table_schema = '{BRONZE_SCHEMA}'
    ORDER BY table_name
""").display()

# Or for a specific table:
# spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.managed_demo").select("tableType", "location").display()

## 7. Common Errors

| Error | Cause | Fix |
|-------|-------|-----|
| Schema mismatch when writing (e.g. `DELTA_METADATA_MISMATCH`) | Column names or types differ from the table schema | Check with `DESCRIBE TABLE`; cast or drop columns. Use `mergeSchema` or `ALTER TABLE ... ADD COLUMN` only when the new column is intended |
| `MERGE` fails because several source rows match one target row | The `ON` key is not unique in the source | Deduplicate the source before `MERGE` |
| Time travel or `RESTORE` to an old version fails | The version is older than the retention window, or `VACUUM` removed its data files | Use a newer version. Retention: `delta.deletedFileRetentionDuration` (data files, default 7 days) and `delta.logRetentionDuration` (history, default 30 days). Time travel is not a backup |

> **Tip:** Run `DESCRIBE HISTORY <table>` to see recent operations — it shows what changed before the error.

## Summary

- Delta = Parquet files + `_delta_log`. It is the default format, gives ACID transactions and is the right choice for tables in Databricks.
- Schema enforcement rejects data that does not match the table. Add columns on purpose with `mergeSchema` or `ALTER TABLE ... ADD COLUMN`; existing rows get `NULL`.
- `INSERT`, `UPDATE`, `DELETE` and `MERGE INTO` change data atomically. MERGE (upsert) is the workhorse of incremental loads.
- `DESCRIBE HISTORY` lists every version. Time travel (`VERSION AS OF n` / `TIMESTAMP AS OF t`) and `RESTORE` work inside the retention window; `VACUUM` deletes old data files, so time travel is not a backup.
- Managed tables: Databricks owns the files, predictive optimization runs `OPTIMIZE`, `VACUUM` and `ANALYZE`, and `UNDROP` works for 7 days. External tables: your team manages the files; predictive optimization does not run on them.

← [Medallion & Ingestion](03_medallion_ingestion.ipynb) | Next: [Lakeflow Jobs →](05_orchestration_jobs.ipynb)

*⊕ BONUS material follows.*

---
> ⊕ **BONUS — Identity Columns and Generated Columns**
>
> Advanced DDL (auto-increment IDs, computed columns) that is not needed at Fundamental level.
---

### Identity and Generated Columns

Delta can fill two kinds of columns for you. A surrogate key is a technical ID with no business meaning.

| Syntax | Description |
|---|---|
| `col BIGINT GENERATED ALWAYS AS IDENTITY` | Auto-incrementing surrogate key — unique and increasing, but not contiguous |
| `col TYPE GENERATED ALWAYS AS (expression)` | Computed column — value derived from other columns at write time |
| `col TYPE GENERATED BY DEFAULT AS IDENTITY` | Like `ALWAYS`, but allows explicit values to be inserted |

In [ ]:
# Drop the demo table if it exists from an earlier run
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.orders_modern")

In [ ]:
# Create table with Identity Column and Generated Column
spark.sql(f"""
CREATE TABLE {CATALOG}.{BRONZE_SCHEMA}.orders_modern (
    order_sk BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate Key
    order_id INT,
    total_amount DOUBLE,
    order_timestamp TIMESTAMP,
    order_date DATE GENERATED ALWAYS AS (CAST(order_timestamp AS DATE)) -- Auto-calculated
) USING DELTA
""")

> **Note:** in a distributed environment like Databricks (Spark/Delta Lake), `GENERATED ALWAYS AS IDENTITY` has specific behaviors. It guarantees **uniqueness** and an **increasing trend**, but does NOT guarantee contiguous numbering.

Now we will insert data. Note that the DataFrame written below omits `order_sk` and `order_date` columns:
- `order_sk`: will be generated automatically (unique number)
- `order_date`: will be calculated based on `order_timestamp`

In [ ]:
# Generate random rows — reduced from 10M to 500K for training cluster safety
from pyspark.sql.functions import expr, rand, concat, lit, round as spark_round, current_timestamp

row_count = 500_000

orders_df = (
    spark.range(row_count)
    .withColumn("order_id", expr("CAST(id AS INT)"))  # Casts the 'id' column from BIGINT to INT
    .withColumn("total_amount", spark_round(rand() * 1000 + 10, 2))
    .withColumn("order_timestamp", current_timestamp())
    .select("order_id", "total_amount", "order_timestamp")
)

orders_df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.orders_modern")

In [ ]:
# order_sk and order_date were filled in by Delta — they were not in the DataFrame
display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.orders_modern")
    .select("order_sk", "order_id", "total_amount", "order_timestamp", "order_date")
    .orderBy("order_sk")
    .limit(10)
)

---
> ⊕ **BONUS — Data Quality with Constraints**
>
> CHECK constraints are an optional task (Task 04) in the Delta workshop.
---

### Data Quality with Constraints

Constraints are data quality rules at table level, as in traditional SQL databases. A write that breaks a rule fails as a whole transaction.

Constraint types:
- `NOT NULL`: enforces the presence of a value
- `CHECK`: enforces any logical condition (e.g., `age > 0`, `customer_id LIKE 'CUST%'`)

Before adding a constraint, the existing rows must satisfy it, so first remove rows without a `customer_id`.
This is a safety step: if the table has no such rows, the `DELETE` changes nothing.

In [ ]:
# Preview the table without rows that have a NULL customer_id
cleaned_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta").na.drop(subset=["customer_id"])
display(cleaned_df.orderBy("customer_id"))

In [ ]:
# Remove rows with a NULL customer_id
spark.sql(f"""
    DELETE FROM {CATALOG}.{BRONZE_SCHEMA}.customers_delta
    WHERE customer_id IS NULL
""")
print("Deleted NULL customer_id rows")

In [ ]:
# Add CHECK constraint: customer_id must start with CUST
try:
    spark.sql(f"""
        ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.customers_delta
        ADD CONSTRAINT valid_customer_id CHECK (customer_id LIKE 'CUST%')
    """)
    print("Constraint 'valid_customer_id' added successfully.")
except Exception as e:
    print(f"Info: {e}")

In [ ]:
# Attempt to insert invalid data (customer_id does not start with CUST)
try:
    spark.sql(f"""
        INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.customers_delta (customer_id, first_name, last_name, email, phone, city, state, country, registration_date, customer_segment)
        VALUES ('INVALID123', 'Bad', 'Customer', 'bad@example.com', '+48 000 000 000', 'Test', 'TS', 'Poland', '2023-01-01', 'Basic')
    """)
except Exception as e:
    print(f"Expected Data Quality error:\n{str(e)[:300]}...")

In [ ]:
# A customer_id that starts with CUST passes the constraint
spark.sql(f"""
  INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.customers_delta (customer_id, first_name, last_name, email, phone, city, state, country, registration_date, customer_segment)
  VALUES ('CUST123', 'Bad', 'Customer', 'bad@example.com', '+48 000 000 000', 'Test', 'TS', 'Poland', '2023-01-01', 'Basic')""")

---
> ⊕ **BONUS — Multi-Statement Transactions**
>
> Good to know for warehousing workloads; not needed in the workshop.
---

### Multi-Statement, Multi-Table Transactions

Every Delta statement is atomic on its own. A transaction groups several statements, on one or more tables, into one atomic unit: all succeed or all roll back.

Transactions are generally available for Unity Catalog managed Delta tables (since July 2026) and managed Iceberg tables (since October 2026); the tables need catalog commits enabled.

Example:

```sql
BEGIN ATOMIC
    INSERT INTO orders VALUES ('ORD-001', 'SKU-9981', 4);
    UPDATE inventory SET quantity_in_stock = quantity_in_stock - 4 WHERE item_sku = 'SKU-9981';
END;
```

Docs: [Transactions](https://learn.microsoft.com/azure/databricks/transactions/)

---
> ⊕ **BONUS — Delta Log Internals (`_delta_log/`)**
>
> A storage-level look at the raw JSON transaction log; not needed to use Delta tables.
---

### Delta Log Internals

See how Delta Lake ensures ACID: every change is written as a JSON file in `_delta_log`.

The Delta Log is a transaction log stored in the `_delta_log/` folder. Each transaction creates a new JSON file containing:
- `add`: Adding a new Parquet file with data
- `remove`: Logical deletion of a file (e.g., during DELETE or OPTIMIZE)
- `commitInfo`: Metadata about the transaction (who, when, what operation)

The cells below show where the table lives and the history that Delta builds from these JSON files.

![Example: Delta Log Internals (Deep Dive)](../../assets/images/f0361af0b2ef43c0a1f61ee1202e2df3.png)

In [ ]:
# Show table location, file count and size
table_details = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.customers_delta")
display(table_details.select("location", "numFiles", "sizeInBytes"))

In [ ]:
# View complete Delta table history
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers_delta"))

---
> ⊕ **BONUS — VACUUM demo (0-hour retention)**
>
> The demo lowers this table's retention safeguard to 0 hours, which you never do in production; the concept is in "VACUUM and Retention" (section 5).
---

### VACUUM and Its Impact on Time Travel

See which versions stay readable after VACUUM removes old data files.

- `VACUUM` deletes data files that the current version no longer uses and that are older than `delta.deletedFileRetentionDuration` (default 7 days). It does not touch `_delta_log`.
- On Databricks Runtime 18.0 and above, and on earlier runtimes for Unity Catalog managed tables, time travel to versions older than `delta.deletedFileRetentionDuration` is blocked even before `VACUUM` runs, and `VACUUM ... RETAIN n HOURS` ignores `n` unless it is 0.
- So in this demo, setting the property to 0 hours already blocks the older versions; `VACUUM` then also deletes their files. The loop below prints the real error for each version.
- The last cell resets the property to the default.

In [ ]:
# Add two more rows — this creates new data files
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VALUES
    (5, 'Eve', 'active', current_timestamp()),
    (6, 'Frank', 'active', current_timestamp())
""")

In [ ]:
# Check current table size and files BEFORE VACUUM
before_vacuum = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")
display(before_vacuum.select("numFiles", "sizeInBytes"))

In [ ]:
# Let's see what versions are available BEFORE vacuum
print("Available versions before VACUUM:")
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo").select("version", "timestamp", "operation"))

In [ ]:
# Lower the retention on this one table (DEMO ONLY)
# In production, NEVER use 0 hours - keep the default 7 days or more!
# VACUUM uses the table property delta.deletedFileRetentionDuration (default 7 days),
# so the demo lowers it on this one table — works on classic clusters and on Serverless
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo
    SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')
""")

In [ ]:
# VACUUM with 0 hours retention (DEMO ONLY)
vacuum_result = spark.sql(f"""
    VACUUM {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo RETAIN 0 HOURS
""")

display(vacuum_result)

In [ ]:
# Check table size AFTER VACUUM
# numFiles / sizeInBytes describe the CURRENT version only, so they do not change:
# VACUUM removes files that only OLD versions still needed.
after_vacuum = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")
display(after_vacuum.select("numFiles", "sizeInBytes"))

In [ ]:
# Turn off the local disk cache first: files read earlier in this notebook are still cached
# on the cluster and could be served from there even though VACUUM deleted them from storage.
# (Serverless does not allow changing this setting)
if not IS_SERVERLESS:
    spark.conf.set("spark.databricks.io.cache.enabled", "false")

In [ ]:
# Now try to read every version and print what happens
for row in spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo").orderBy("version").collect():
    try:
        # read the rows themselves — COUNT(*) alone is answered from the Delta log without opening data files
        rows = spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VERSION AS OF {row.version}").collect()
        print(f"Version {row.version:>2}  {row.operation:<25} readable      ({len(rows)} rows)")
    except Exception as e:
        # print the real reason: retention block or missing files, depending on the runtime
        print(f"Version {row.version:>2}  {row.operation:<25} NOT readable  {type(e).__name__}: {str(e)[:150]}")

In [ ]:
# Turn the local disk cache back on
if not IS_SERVERLESS:
    spark.conf.set("spark.databricks.io.cache.enabled", "true")

In [ ]:
# Reset the retention property to the default (7 days)
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo
    UNSET TBLPROPERTIES IF EXISTS ('delta.deletedFileRetentionDuration')
""")

print("delta.deletedFileRetentionDuration removed — the table uses the default (7 days) again")

---
> ⊕ **BONUS — External Tables**
>
> External tables need an External Location in Unity Catalog, set up by an admin.
---

### External Table

You specify `LOCATION`, and the table points to files in your own storage (ADLS, S3, GCS).
In Unity Catalog, external tables require an External Location (managed by an admin).

```sql
CREATE TABLE catalog.schema.external_table
USING DELTA
LOCATION 'abfss://container@storageaccount.dfs.core.windows.net/path/to/data'
```

> **Note:** `DROP TABLE` removes only the metadata. The files still exist in storage — this is the intended behaviour for external tables.

The demo below runs only when you set `EXTERNAL_LOCATION_PATH` to a path you can write to.

In [ ]:
# Set this to a path inside an External Location you can write to, for example
# "abfss://<container>@<storage-account>.dfs.core.windows.net/external/external_demo"
EXTERNAL_LOCATION_PATH = ""

In [ ]:
# Drop the demo table if it exists from an earlier run
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.external_demo")

In [ ]:
# Create the external table (skipped when the path is empty)
if not EXTERNAL_LOCATION_PATH:
    print("Skipped: EXTERNAL_LOCATION_PATH is empty. External tables need an External Location set up by an admin.")
else:
    try:
        spark.sql(f"""
            CREATE TABLE {CATALOG}.{BRONZE_SCHEMA}.external_demo
            (id INT, name STRING, created_at TIMESTAMP)
            USING DELTA
            LOCATION '{EXTERNAL_LOCATION_PATH}'
        """)
        print("External table created.")
    except Exception as e:
        print(f"Could not create the external table: {type(e).__name__}: {str(e)[:200]}")

In [ ]:
# Contrast: a path-based Delta write. It creates Delta files in a Volume but registers NO table
# in Unity Catalog — it is not an external table. Prefer tables; Volumes are meant for non-tabular files.
external_path = f"{DATASET_PATH}/external/customers_delta"
customers_df.write.format("delta").mode("overwrite").save(external_path)

---
> ⊕ **BONUS — More Delta Errors (concurrency, manual file changes)**
>
> These errors come from concurrent pipelines or manual changes in storage, not from beginner work.
---

| Error | Cause | Fix |
|-------|-------|-----|
| `ConcurrentAppendException` (and other concurrent-modification errors) | Two jobs changed the same table data at the same time | Retry the write, or let concurrent jobs touch separate data (disjoint `WHERE` conditions). The stricter `Serializable` isolation level causes more conflicts, not fewer |
| `Cannot overwrite a path that is also being read` | A path-based write of plain files that reads and overwrites the same path | Write Delta tables by name; for plain files write to a new location first |
| `DELTA_MISSING_TRANSACTION_LOG` | `_delta_log/` deleted manually | Never delete files inside a table directory — use `DROP TABLE` |